## Load Necessary Packages

In [ ]:
# Reminder to upload 'CFTR2 variants genomic locations_2023.xlsx' for exonic CFTR variant data

In [ ]:
#!pip install openpyxl
#!pip install mpl_axes_aligner
#!pip install session_info
#!pip install shap
#!pip install graphviz

In [ ]:
# Load Packages
import pandas
import os
import numpy as np
import math
import openpyxl
import mpl_axes_aligner
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import json
import plotly
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.patches as mpatches
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from platform import python_version
from sklearn import __version__ as sklearn_version

# Get Package Versions
print(f"python version: {python_version()}")
print(f"pandas version: {pandas.__version__}")
print(f"os version: {os.name}")
print(f"numpy version: {np.__version__}")
#print(f"hail version: {hl.version()}") # If you get an error remove this line, Hail is loaded afterwords
print(f"openpyxl version: {openpyxl.__version__}")
# print(f"mpl_axes_aligner version: {mpl_axes_aligner.__version__}") has no version 
print(f"matplotlib version: {matplotlib.__version__}")
print(f"json version: {json.__version__}")
print(f"plotly version: {plotly.__version__}")
print(f"sklearn version: {sklearn_version}")

## Load Clinical/Demographic data of the Cohort from the All of Us Research Database

In [ ]:
# This query represents dataset "Only pwCF analysis" for domain "person" and was generated for All of Us Controlled Tier Dataset v7
dataset_51941639_person_sql = """
    SELECT
        person.person_id,
        person.gender_concept_id,
        p_gender_concept.concept_name as gender,
        person.birth_datetime as date_of_birth,
        person.race_concept_id,
        p_race_concept.concept_name as race,
        person.ethnicity_concept_id,
        p_ethnicity_concept.concept_name as ethnicity,
        person.sex_at_birth_concept_id,
        p_sex_at_birth_concept.concept_name as sex_at_birth 
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.person` person 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_gender_concept 
            ON person.gender_concept_id = p_gender_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_race_concept 
            ON person.race_concept_id = p_race_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_ethnicity_concept 
            ON person.ethnicity_concept_id = p_ethnicity_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_sex_at_birth_concept 
            ON person.sex_at_birth_concept_id = p_sex_at_birth_concept.concept_id  
    WHERE
        person.PERSON_ID IN (
            SELECT
                distinct person_id  
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
            WHERE
                cb_search_person.person_id IN (
                    SELECT
                        person_id 
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
                    WHERE
                        has_whole_genome_variant = 1 
                ) 
                AND cb_search_person.person_id IN (
                    SELECT
                        criteria.person_id 
                    FROM
                        (SELECT
                            DISTINCT person_id,
                            entry_date,
                            concept_id 
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                        WHERE
                            (
                                concept_id IN(
                                    SELECT
                                        DISTINCT c.concept_id 
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                                    JOIN
                                        (
                                            SELECT
                                                CAST(cr.id as string) AS id       
                                            FROM
                                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                                            WHERE
                                                concept_id IN (45769146, 254320, 45768915, 434615, 193174, 441267, 4143529, 4144583)       
                                                AND full_text LIKE '%_rank1]%'      
                                        ) a 
                                            ON (
                                                c.path LIKE CONCAT('%.',
                                            a.id,
                                            '.%') 
                                            OR c.path LIKE CONCAT('%.',
                                            a.id) 
                                            OR c.path LIKE CONCAT(a.id,
                                            '.%') 
                                            OR c.path = a.id) 
                                        WHERE
                                            is_standard = 1 
                                            AND is_selectable = 1
                                        ) 
                                        AND is_standard = 1 
                                )
                            ) criteria 
                        ) )"""

dataset_51941639_person_df = pandas.read_gbq(
    dataset_51941639_person_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

dataset_51941639_person_df.head(5)

## Load and manipulate the extracted Genomic Data for analysis

In [ ]:
# Set DATASET_51941639_VCF_DIR to where the genomic data is extracted
%env DATASET_51941639_VCF_DIR= # insert google cloud bucket with data here

In [ ]:
import os
import subprocess

# The extraction workflow outputs a manifest file upon completion.
manifest_file = os.environ['DATASET_51941639_VCF_DIR'] + '/manifest.txt'

assert subprocess.run(['gsutil', '-q', 'stat', manifest_file]).returncode == 0, (
  "!" * 100 + "\n\n" +
  "VCF extraction has not completed.\n" +
  "Please monitor the extraction sidepanel for completion before continuing.\n\n" +
  "!" * 100
)

print("VCF extraction has completed, continuing")


In [ ]:
# Confirm Spark is installed.
try:
    import pyspark
except ModuleNotFoundError:
    print("!" * 100 + "\n\n"
          "In the Researcher Workbench, Hail can only be used on a Dataproc cluster.\n"
          "Please use the 'Cloud Analysis Environment' side panel to update your runtime compute type.\n\n" +
          "!" * 100)

# Initialize Hail
import hail as hl
import os
from hail.plot import show

hl.init(default_reference='GRCh38', global_seed = 0)
hl.plot.output_notebook()

In [ ]:
# Create Hail Matrix table
# This can take a few hours for a dataset with hundreds of participants
workspace_bucket = os.environ['WORKSPACE_BUCKET']
vcf_dir = os.environ['DATASET_51941639_VCF_DIR']
hail_matrix_table_gcs = f'{workspace_bucket}/dataset_51941639.mt'
#hl.import_vcf(f'{vcf_dir}/*.vcf.gz', force_bgz=True, array_elements_required=False).write(hail_matrix_table_gcs)
# unhash above line when running for the first time 


In [ ]:
# Read Hail Matrix table
mt_51941639 = hl.read_matrix_table(hail_matrix_table_gcs)

In [ ]:
# Indicate the chromosomal location of each CFTR intron and extract variants belonging to each intron
# chromosomal locations based on GRCh38 genome assembly of the ENST00000003084.11 CFTR Transcript 

Intron_1 = ['chr7:117480147-117504252']
Intron_2 = ['chr7:117504363-117509033']
Intron_3 = ['chr7:117509142-117530898']
Intron_4 = ['chr7:117531114-117534275']
Intron_5 = ['chr7:117534365-117535247']
Intron_6 = ['chr7:117535411-117536547']
Intron_7 = ['chr7:117536673-117540099']
Intron_8 = ['chr7:117540346-117542015']
Intron_9 = ['chr7:117542108-117548640']
Intron_10 = ['chr7:117548823-117559463']
Intron_11 = ['chr7:117559655-117587738']
Intron_12 = ['chr7:117587833-117590352']
Intron_13 = ['chr7:117590439-117591933']
Intron_14 = ['chr7:117592657-117594929']
Intron_15 = ['chr7:117595058-117602825']
Intron_16 = ['chr7:117602863-117603531']
Intron_17 = ['chr7:117603782-117606673']
Intron_18 = ['chr7:117606753-117610518']
Intron_19 = ['chr7:117610669-117611580']
Intron_20 = ['chr7:117611808-117614612']
Intron_21 = ['chr7:117614713-117627521']
Intron_22 = ['chr7:117627770-117642437']
Intron_23 = ['chr7:117642593-117652841']
Intron_24 = ['chr7:117652931-117664687']
Intron_25 = ['chr7:117664860-117665458']
Intron_26 = ['chr7:117665564-117666907']

# Create a dictionary with this data
mt_dict = {}

for i in range(1, 27):
    filtered_mt = hl.filter_intervals(
        mt_51941639,
        [hl.parse_locus_interval(x)
         for x in globals()[f'Intron_{i}']]
    )
    mt_dict[f'mt{i}'] = filtered_mt


In [ ]:
# Read in Excel file containing CFTR variant info, from the CFTR2 database, as a pandas dataframe
excel_df = pandas.read_excel(f'{workspace_bucket}/data/CFTR2 variants genomic locations_2023.xlsx')

# Code to extract the starting position of the first CFTR variant and the ending position of the last CFTR variant
# This allows us to have a targetted region from which we will query the variants that individuals in the 
# cohort have
start_pos = min(excel_df['grch38_pos'])
start_pos_chr = min(excel_df['grch38_chr'])
end_pos = max(excel_df['grch38_pos'])+ 1 # 1 because last variant is SNP, change accordingly (e.g if the alt
# allele changed 3 nucleotides then + 3)
end_pos_chr = max(excel_df['grch38_chr'])

# Extract Variants from selected region
region_to_analyze = [f'chr{start_pos_chr}:{start_pos}-chr{end_pos_chr}:{end_pos}']
mt_annotate = hl.filter_intervals(
    mt_51941639,
    [hl.parse_locus_interval(x,)
     for x in region_to_analyze])

## Filter the cohort on the basis of relatedness

In [ ]:
# Collecting Relatedness Info
auxiliary_path = "gs://fc-aou-datasets-controlled/v7/wgs/short_read/snpindel/aux"
relatedness_path= f'{auxiliary_path}/relatedness/relatedness.tsv'
relatedness=hl.import_table(relatedness_path)
relatedness_df = relatedness.to_pandas()
people = mt_annotate.col.s.collect()

# Collecting individuals which are related to one another (kinship score >= 0.125)
relatedness_df = relatedness_df[relatedness_df['kin'].astype(float) >= 0.125]
filtered_relatedness_df = relatedness_df[(relatedness_df['i.s'].isin(people)) & (relatedness_df['j.s'].isin(people))]

# Removing data (across all introns) for one of the individual in each related pairs 
for x in filtered_relatedness_df['i.s']:
    mt_annotate = mt_annotate.filter_cols(mt_annotate.s != x) 

# Uploading Hail Matrix Table GT values (all introns), downloading and reading them in as a pandas dataframe 
mt_annotate.GT.export(f'{workspace_bucket}/data/annotateGT.tsv')
df_annotations = pandas.read_csv(f'{workspace_bucket}/data/annotateGT.tsv', sep= '\t')


## Annotate individuals within the cohort by their CFTR variant(s) and their deemed CF-Status

In [ ]:
## Do not need to run again if results have been uploaded 

# create cdna column in df_annotations 
#df_annotations['alleles'] = df_annotations['alleles'].apply(lambda x: json.loads(x))

#df_annotations['cdna_name']= ""
#df_annotations['legacy_name']= ""
#df_annotations['cftr2_annotation']= ""

#excel_drop = []

# Filter excel sheet so it only contains variants found in the individual data (not really necessary)
#for x_index, x_row in excel_df.iterrows():
#    match = False 
#    for y_index, y_row in df_annotations.iterrows():
#        if ((f"chr{x_row['grch38_chr']}:{x_row['grch38_pos']}" == y_row['locus']) & 
#            (str(x_row['grch38_ref']) == (y_row['alleles'])[0]) &
#                (str(x_row['grch38_alt']) == (y_row['alleles'])[1])):
#            match = True
#            break
#   if not match:
#        excel_drop.append(x_index)

#excel_df.drop(excel_drop, inplace = True)

#variant_drop = []

# Filtering individual data so it only contains variants in the excel sheet 
#for y_index, y_row in df_annotations.iterrows():
#    match = False
#    for x_index, x_row in excel_df.iterrows():
#        if ((f"chr{x_row['grch38_chr']}:{x_row['grch38_pos']}" == y_row['locus']) & 
#            (str(x_row['grch38_ref']) == (y_row['alleles'][0])) &
#                (str(x_row['grch38_alt']) == y_row['alleles'][1])):
           
#            df_annotations.at[y_index,'cdna_name'] = excel_df.at[x_index,'cdna_name']
#            df_annotations.at[y_index,'legacy_name'] = excel_df.at[x_index,'all_legacy_names']
#            df_annotations.at[y_index, 'cftr2_annotation'] = excel_df.at[x_index, 'cftr2_annotation']
            
#            match = True
#            break 
                
#    if not match:
#        variant_drop.append(y_index)

#df_annotations.drop(variant_drop, inplace = True) 
#df_annotations2 = df_annotations.copy(deep=True)

# Upload files online
#df_annotations.export(f'{workspace_bucket}/data/df_annotations.csv')
#df_annotations2.export(f'{workspace_bucket}/data/df_annotations2.csv')

# Replacing call values with annotation for the presence or lack of mutation 
#df_annotations = df_annotations.replace('0/0', "No Mutation")
#df_annotations = df_annotations.replace('0/1',"Heterozygous Mutation")
#df_annotations = df_annotations.replace('1/0',"Heterozygous Mutation")
#df_annotations = df_annotations.replace('1/1',"Homozygous Mutation")             
        

In [ ]:
# Download and manipulate the data frame to be more presentable

df_annotations2 = pandas.read_csv(f'{workspace_bucket}/data/df_annotations2.csv')
df_annotations = pandas.read_csv(f'{workspace_bucket}/data/df_annotations.csv')
df_annotations.insert(2,"cdna_name",df_annotations.pop("cdna_name"))
df_annotations.insert(3,"legacy_name",df_annotations.pop("legacy_name"))
df_annotations.insert(4,"cftr2_annotation",df_annotations.pop("cftr2_annotation"))
df_annotations

# Calculate summary statistics of each variant in the cohort
df_annotation_summary = df_annotations.iloc[:,0:4]
df_annotation_summary["Homozygous Mutation %"] = df_annotations.apply(lambda row: 
                                                                      (row.str.count("Homozygous Mutation").sum()/
                                                                       (df_annotations.shape[1]-5))*100, axis=1)
df_annotation_summary["Heterozygous Mutation %"] = df_annotations.apply(lambda row: 
                                                                      (row.str.count("Heterozygous Mutation").sum()/
                                                                       (df_annotations.shape[1]-5))*100, axis=1)
df_annotation_summary["No Mutation %"] = df_annotations.apply(lambda row: 
                                                                      (row.str.count("No Mutation").sum()/
                                                                       (df_annotations.shape[1]-5))*100, axis=1)
df_annotation_summary["NaN %"] = df_annotations.apply(lambda row: 
                                                                      (row.isna().sum()/
                                                                       (df_annotations.shape[1]-5))*100, axis=1)

# Export Variant annotation 
df_annotation_summary.to_excel(f'{workspace_bucket}/data/Genotype Files/pwCF/Total Variant Annotation.xlsx')


In [ ]:
# Create a plot of the annotation results 

fig, ax = plt.subplots(figsize=(24,12))
y1 = df_annotation_summary["No Mutation %"]
y2 = df_annotation_summary["Heterozygous Mutation %"]
y3 = df_annotation_summary["Homozygous Mutation %"]
y4 = df_annotation_summary["NaN %"]

ax.bar(df_annotation_summary["legacy_name"], y1, color= 'beige', edgecolor= 'black', label="No Mutation %")

ax.bar(df_annotation_summary["legacy_name"], y2, bottom = y1, color= 'violet', edgecolor = 'black', 
       label="Heterozygous Mutation %")

ax.bar(df_annotation_summary["legacy_name"],df_annotation_summary["Homozygous Mutation %"], bottom = y1 + y2,
       color= 'black', edgecolor = 'black', label="Homozygous Mutation %")

ax.bar(df_annotation_summary["legacy_name"],df_annotation_summary["NaN %"],bottom = y1 + y2 + y3,
       color= 'red', edgecolor = 'black', label="NaN %")

ax.set_xlabel("Legacy Variant Name")
ax.set_ylabel("%")
ax.set_title("Variant Profiling of the n=307 pwCF Cohort")
ax.legend(loc = 'lower right')
plt.xticks(rotation=90)
size = 12
params = {'legend.fontsize': 'large',
          'figure.figsize': (20,8),
          'axes.labelsize': size*1.25,
          'axes.titlesize': size*1.25,
          'xtick.labelsize': size*0.75,
          'ytick.labelsize': size*0.75,
          'axes.titlepad': 15}
plt.rcParams.update(params)
plt.show()


In [ ]:
# Assess the top 4 mutations in greater detail

# Filter for the top 4 mutations
df_annotations_cropped = df_annotations[(df_annotations["legacy_name"] == "125G/C|5UTR-8G->C") |
                                        (df_annotations["legacy_name"] == "T854T|2694T/G|T854T (2694T/G)") |
                                        (df_annotations["legacy_name"] == "F508del|1653delCTT|[delta]F508|^F508|dF508") |
                                        (df_annotations["legacy_name"] ==  "V470M")]

# Make a table for annotation purposes 
df_annotations_cropped_transposed = df_annotations_cropped.iloc[:,4:].transpose().copy(deep=True)
df_annotations_cropped_transposed.columns = ["125G/C|5UTR-8G->C","V470M","F508del|1653delCTT|[delta]F508|^F508|dF508",
                                            "T854T|2694T/G|T854T (2694T/G)"]
df_annotations_cropped_transposed = df_annotations_cropped_transposed[1:].reset_index(drop=True)
#df_annotations_cropped_transposed = df_annotations_cropped_transposed.replace({'Homozygous Mutation': 2, 
#                                                                              'Heterozygous Mutation': 1,
#                                                                              'No Mutation': 0})
#df_annotations_cropped_transposed = df_annotations_cropped_transposed.fillna("None")


In [ ]:
# Create a DataFrame listing the mutation for each individual and whether it is CF-Causing or not 

individual_mutations = pandas.DataFrame()
for x in df_annotations.columns[5:] :
    individual_mutations.at[x,'Individual'] = x
    mutation_number = 1
    F508del = 0
    CF_causing = 0 
    Varying = 0
    Unknown = 0
    Non_CF_causing = 0
    Left = ''
    Right = ''
    Labelling_Group = ''
    condition_status = False
    
    for y in range(len(df_annotations)):
        if ((df_annotations[x][y] != "No Mutation") and (not pandas.isna(df_annotations[x][y]))):
            if df_annotations[x][y] == "Heterozygous Mutation":
                individual_mutations.at[x,f"Mutation {mutation_number}"] = df_annotations['legacy_name'][y]

            if df_annotations[x][y] == "Homozygous Mutation":
                individual_mutations.at[x,f"Mutation {mutation_number}"] = df_annotations['legacy_name'][y]
                individual_mutations.at[x,f"Mutation {mutation_number} CFTR2 Annotation"] = df_annotations['cftr2_annotation'][y]
                mutation_number += 1
                individual_mutations.at[x,f"Mutation {mutation_number}"] = df_annotations['legacy_name'][y]
    
            individual_mutations.at[x,f"Mutation {mutation_number} CFTR2 Annotation"] = df_annotations['cftr2_annotation'][y]
            mutation_number +=1
            
        if (not pandas.isna(df_annotations[x][y])):
            if df_annotations['legacy_name'][y] == 'F508del|1653delCTT|[delta]F508|^F508|dF508':
                if ((df_annotations2[x][y] == '0/1') or (df_annotations2[x][y] == '1/0')) :
                    F508del = 1
                if df_annotations2[x][y] == '1/1':
                    F508del = 2
            
            if ((df_annotations['cftr2_annotation'][y] == 'CF-causing') and 
                (df_annotations['legacy_name'][y] !='F508del|1653delCTT|[delta]F508|^F508|dF508')):
                if ((df_annotations2[x][y] == '0/1') or (df_annotations2[x][y] == '1/0')):
                    CF_causing += 1
                if df_annotations2[x][y] == '1/1':
                    CF_causing += 2 

            if (df_annotations['cftr2_annotation'][y] == 'Varying clinical consequence'):
                if ((df_annotations2[x][y] == '0/1') or (df_annotations2[x][y] == '1/0')):
                    Varying += 1
                if df_annotations2[x][y] == '1/1':
                    Varying += 2

            if (df_annotations['cftr2_annotation'][y] == 'Unknown significance'):
                if ((df_annotations2[x][y] == '0/1') or (df_annotations2[x][y] == '1/0')):
                    Unknown += 1
                if df_annotations2[x][y] == '1/1':
                    Unknown += 2

            if (df_annotations['cftr2_annotation'][y] == 'Non CF-causing'):
                if ((df_annotations2[x][y] == '0/1') or (df_annotations2[x][y] == '1/0')):
                    Non_CF_causing += 1
                if df_annotations2[x][y] == '1/1':
                    Non_CF_causing += 2
                    
    if F508del == 1:
        Left = 'F508del'
    if F508del == 2:
        Left = 'F508del'
        Right = 'F508del'
            
    if CF_causing == 1:
        if (Left != '' and Right == ''):
            Right = 'CF-Causing'
        if Left == '':
            Left = 'CF-Causing'
    if CF_causing > 1:
        if Left == '':
            Left = 'CF-Causing'
        if Right == '':
            Right = 'CF-Causing'
            
    if Varying == 1:
        if (Left != '' and Right == ''):
            Right = 'Varying Clinical Significance'
        if Left == '':
            Left = 'Varying Clinical Significance'
    if Varying > 1:
        if Left == '':
            Left = 'Varying Clinical Significance'
        if Right == '':
            Right = 'Varying Clinical Significance'
            
    if Unknown == 1:
        if (Left != '' and Right == ''):
            Right = 'Unknown Significance'
        if Left == '':
            Left = 'Unknown Significance'
    if Unknown > 1:
        if Left == '':
            Left = 'Unknown Significance'
        if Right == '':
            Right = 'Unknown Significance'

    if Non_CF_causing == 1:
        if (Left != '' and Right == ''):
            Right = 'Non CF-Causing'
        if Left == '':
            Left = 'Non CF-Causing'
    if Non_CF_causing > 1:
        if Left == '':
            Left = 'Non CF-Causing'
        if Right == '':
            Right = 'Non CF-Causing'
            
    if Left == '':
        Left = 'None'
    if Right == '':
        Right = 'None'
                
    individual_mutations.at[x,"Classification"] = Left + '/' + Right
        
    if ((individual_mutations.at[x,"Classification"] == "F508del/F508del") or
        (individual_mutations.at[x,"Classification"] == "F508del/CF-Causing") or
        (individual_mutations.at[x,"Classification"] == "CF-Causing/CF-Causing") or
        (individual_mutations.at[x,"Classification"] == "F508del/Varying Clinical Significance") or
        (individual_mutations.at[x,"Classification"] == "CF-Causing/Varying Clinical Significance")):
        condition_status = True
        individual_mutations.at[x,"CF Designation"] = 'CF'
        
    if not condition_status:
        individual_mutations.at[x,"CF Designation"] = 'Non-CF'
                     

individual_mutations = individual_mutations.fillna('')
individual_mutations.insert(1,"Classification",individual_mutations.pop("Classification"))
individual_mutations.insert(2,"CF Designation",individual_mutations.pop("CF Designation"))
individual_mutations
                                                                

## Extract Race and Ancestry data

In [ ]:
# Create two lists separating individuals with respect to their mutation status
individuals_with_mutations = []
individuals_with_mutations_key = []
individuals_without_mutations = []
individuals_without_mutations_key = []
for x in range(len(individual_mutations)):
    if individual_mutations['Classification'][x] == "None/None":
        individuals_without_mutations.append(individual_mutations['Individual'][x])
        individuals_without_mutations_key.append(x)
    else:
        individuals_with_mutations.append(individual_mutations['Individual'][x])
        individuals_with_mutations_key.append(x)

# Extract Ancestry data 
auxiliary_path = "gs://fc-aou-datasets-controlled/v7/wgs/short_read/snpindel/aux"
ancestry_path = f'{auxiliary_path}/ancestry'
ancestry_pred_path = f'{ancestry_path}/ancestry_preds.tsv'
ancestry_pred = hl.import_table(ancestry_pred_path,
                               key="research_id", 
                               impute=True, 
                               types={"research_id":"tstr","pca_features":hl.tarray(hl.tfloat)})
ancestry_df_temp=ancestry_pred.select('ancestry_pred').to_pandas()

# Filter Ancestry data for all individuals being analyzed, split based on mutation status
ancestry_df_mutations = ancestry_df_temp[ancestry_df_temp['research_id'].isin(individuals_with_mutations)]
ancestry_df_no_mutations = ancestry_df_temp[ancestry_df_temp['research_id'].isin(individuals_without_mutations)]


In [ ]:
# Creating Data Frame which contains self-reported race data for all individuals being analyzed 
df_ht = dataset_51941639_person_df[['person_id','race']]
df_ht['person_id'] = df_ht['person_id'].astype(str)
df_ht = hl.Table.from_pandas(df_ht, key = 'person_id')
df_ht.show()

## Extract Genotype Call Data based on Intronic Region and perform Quality Control
Analysis from this point on needs to be repeated for each Intron 

In [ ]:
# Choose Intronic Region to assess (you can change the intron value below to switch what intron you want to assess)
intron=1
mt = mt_dict['mt1']

# Obtain ancestry prediction information from all of us and annotate columns with it, the self reported 
# information was obtained by the data frame created in the first step
ancestry_path = f'{auxiliary_path}/ancestry'
ancestry_pred_path = f'{ancestry_path}/ancestry_preds.tsv'
ancestry_pred = hl.import_table(ancestry_pred_path,
                               key="research_id", 
                               impute=True, 
                               types={"research_id":"tstr","pca_features":hl.tarray(hl.tfloat)})

# Annotate Individuals Based on Genomic Ancestry
mt = mt.annotate_cols(ancestry_pred = ancestry_pred[mt.s])

# Annotate Individuals Based on Self-Reported Race
mt = mt.annotate_cols(race = df_ht[mt.s])

In [ ]:
# Collecting Relatedness Info
relatedness=hl.import_table(relatedness_path)
relatedness_df = relatedness.to_pandas()
people = mt.col.s.collect()

# Collecting individuals which are related to one another (no filter in terms of relatedness score set)
relatedness_df = relatedness_df[relatedness_df['kin'].astype(float) >= 0.125]
filtered_relatedness_df = relatedness_df[(relatedness_df['i.s'].isin(people)) & (relatedness_df['j.s'].isin(people))]
print(filtered_relatedness_df)                                

# Removing data (in chosen intron) for one of the individuals in each related pairs 
for x in filtered_relatedness_df['i.s']:
    mt = mt.filter_cols(mt.s != x)    

# Check details of the variants found within each intron, in greater detail
hl.summarize_variants(mt)

In [ ]:
# Split Multi-Allelic Loci into multiple Bi-Allelic Loci 
mt = hl.split_multi_hts(mt)

hl.summarize_variants(mt)

# Convert entries with a genotype quality of below 20 to NA
mt_filtered = mt.annotate_entries(GQ=hl.if_else((mt.GQ < 20), hl.missing(mt.GQ.dtype), mt.GQ))

# Remove rows which have a genotype quality of NA in over 10% of columns 
cond = hl.agg.count_where(hl.is_missing(mt_filtered.GQ))
mt_filtered2 = mt_filtered.filter_rows((cond/mt_filtered.count_cols()) <= 0.1)

# check how many variants remain
hl.summarize_variants(mt_filtered2)

In [ ]:
# Annotate variants with major-minor allele frequencies, etc. 
mt_filtered2=hl.variant_qc(mt_filtered2)

# Filter rows where the dominant allele is present less than 95% of the time or minor alleles is present more than
# 5% of the time
mt_filtered2 = mt_filtered2.filter_rows(mt_filtered2.variant_qc.AF[1] > 0.05)

# Check allele frequencies of remaining variants
mt_filtered2.variant_qc.AF[1].show()

# check how many variants remain
hl.summarize_variants(mt_filtered2)

In [ ]:
# Create a column which indicates how many GT values are defined vs. missing in the form [missing, defined], 
# for each row, for PCA it is important to make sure there are no missing values
mt_filtered2 = mt_filtered2.annotate_rows(counter=[hl.agg.sum(hl.is_missing(mt_filtered2.GT)) , hl.agg.sum
                                                              (hl.is_defined(mt_filtered2.GT))])


In [ ]:
# Turn call values into integers
# Let the integer be an average of the call values from both chromosomes. E.g: 0/0 = 0, 0/1 and 1/0 = 0.5, 1/1 =1 etc. 
# All missing values are temporarily given the the value: 0
def append_genotype_call(genotype):
    genotype = hl.str(genotype).split('/')
    left_side = hl.int32(genotype[0])
    right_side = hl.int32(genotype[1])
    
    geno = (left_side+right_side)/2
    
    return hl.float(geno)

mt_filtered2 = mt_filtered2.annotate_entries(intermediary_GT=hl.or_else(append_genotype_call(mt_filtered2.GT), 
                                                                 hl.float(0)))

In [ ]:
# Sum all the values in the row, so that the mean of the row can be calculated
mt_filtered2 = mt_filtered2.annotate_rows(sum_GT=hl.agg.sum(mt_filtered2.intermediary_GT))

#Determine the mean of the row by dividing the sum of the row with the amount of defined values in a row
def impute_call(genotype):
    genotype = mt_filtered2.sum_GT/(mt_filtered2.counter[1])
    return hl.float(genotype)

# Instead of setting missing values to 0, set the missing values of each row to the calculated mean of the row
mt_filtered3 = mt_filtered2.annotate_entries(intermediary_GT2=hl.or_else(append_genotype_call(mt_filtered2.GT)
                                                                   ,impute_call(mt_filtered2.GT)))

# Create a parallel dataframe which still has missing values, which will be used for machine learning (ML),
# as XGBoost machine learning is able to adjust to missing values (unlike PCA)
mt_filtered4 = mt_filtered2.annotate_entries(ML=append_genotype_call(mt_filtered2.GT))

In [ ]:
# Create an ordered dataframe of self-reported race for scikit PCA and TSNE 
# which outputs scores in preserved matrix table column order [IMPORTANT FOR LABELLING DOWNSTREAM]

# Code to ensure the random switching of person_id in dataset_51941639_person_df does not happen (set to numeric)
dataset_51941639_person_df['person_id'] = dataset_51941639_person_df['person_id'].astype(int)

# First collect the column names (individuals) from the matrix table and turn it into a list
struct_list2 = mt_filtered3.col.collect()
individuals_list2 = pandas.Series([np.int64(int(s.s)) for s in struct_list2])

# Filter the pandas data frame with demographics of each individual and filter for only the individuals
# remaining on the list - sometimes data_519416
filtered_annotation_df2 = dataset_51941639_person_df[dataset_51941639_person_df['person_id'].isin(individuals_list2)].copy()

# Organize individuals based on the matrix table columns order, so that points can be annotated with race
# accurately in the PCA plot
ordered_df2 = filtered_annotation_df2.set_index('person_id').loc[individuals_list2].reset_index()

In [ ]:
# Create an ordered dataframe of computed genetic ancestry for Scikit PCA and TSNE 
# which outputs scores in preserved matrix table column order [IMPORTANT FOR LABELLING DOWNSTREAM]
ancestry_df = pandas.DataFrame(np.array(mt_filtered3.ancestry_pred.ancestry_pred_other.collect()))
ancestry_df = ancestry_df.rename(columns={0: 'Ancestry_Pred'})
#print(ancestry_df)

# This is an ordered dataframe of computed genetic ancestry (without the "other" category) for Scikit PCA and TSNE 
# which outputs scores in preserved matrix table column order

ancestry_no_other_df = pandas.DataFrame(np.array(mt_filtered3.ancestry_pred.ancestry_pred.collect()))
ancestry_no_other_df = ancestry_no_other_df.rename(columns={0: 'Ancestry_Pred'})
#print(ancestry_no_other_df)

## Demographic Data for the Cohort

In [ ]:
# Create a DataFrame with demographic information of each patient

ancestry_no_other_df_temp = ancestry_no_other_df
ancestry_no_other_df_temp.reset_index(drop=True, inplace = True)

individual_mutations_temp = individual_mutations['Individual']
individual_mutations_temp.reset_index(drop=True, inplace = True)

df_demographics = pandas.concat([individual_mutations_temp, ancestry_no_other_df_temp,],axis=1,
                                ignore_index = True)

df_demographics.columns = ['person_id', 'Genomic Race']
dataset_51941639_person_df['person_id'] = dataset_51941639_person_df['person_id'].astype(str)
df_demographics = df_demographics.merge(dataset_51941639_person_df, on = 'person_id')

# Calculate the age for each individual (based on year-month-day)
df_demographics['age'] = ''
today_day = 18
today_month = 7
today_year = 2024

for x in range(len(df_demographics)):
    split = df_demographics['date_of_birth'][x].strftime("%Y-%m-%d").split('-')
    year = int(split[0])
    month = int(split[1])
    day = int(split[2])
    
    if today_day >= day:
        month -= 1
    
    if today_month <= month:
        year += 1
    
    age = today_year - year
    
    df_demographics['age'][x] = age 
    
df_demographics['date_of_birth'] = df_demographics['date_of_birth'].apply(lambda x: x.strftime("%Y-%m-%d"))

        

In [ ]:
# Create a list of classes based on CFTR Classifications and the individuals that fall within each class

List_A = []
List_B = []
List_C = [] 
List_D = []
List_E = []

for x in range(len(individual_mutations)):
    if individual_mutations['Classification'][x] == 'F508del/F508del':
        List_C.append(x)
    elif ((individual_mutations['Classification'][x] == 'F508del/CF-Causing') or
        (individual_mutations['Classification'][x] == 'F508del/Varying Clinical Significance')):
        List_A.append(x)
    elif ((individual_mutations['Classification'][x] == 'CF-Causing/CF-Causing') or
        (individual_mutations['Classification'][x] == 'CF-Causing/Varying Clinical Significance')):
        List_B.append(x)
    elif (individual_mutations['Classification'][x] == 'None/None'):
        List_E.append(x)
    else:
        List_D.append(x)
        


## Labelling Individuals in the cohorts, for plotting 

In [ ]:
# For each individual, indicate their status for the top 4 CFTR mutations and give them a labelling category
# combining their status for these 4 mutations and ancestry

#for x, x_index in df_annotations_cropped_transposed.iterrows():
#    value = ''
#    if (x_index["125G/C|5UTR-8G->C"] == "Heterozygous Mutation"):
#        value += 'a'
#    if (x_index["125G/C|5UTR-8G->C"] == "Homozygous Mutation"):
#        value += 'A'

#    if (x_index["V470M"] == "Heterozygous Mutation"):
#        value += 'b'
#    if (x_index["V470M"] == "Homozygous Mutation"):
#        value += 'B'

#    if (x_index["F508del|1653delCTT|[delta]F508|^F508|dF508"] == "Heterozygous Mutation"):
#        value += 'c'
#    if (x_index["F508del|1653delCTT|[delta]F508|^F508|dF508"] == "Homozygous Mutation"):
#       value += 'C'

#    if (x_index["T854T|2694T/G|T854T (2694T/G)"] == "Heterozygous Mutation"):
#        value += 'd'
#    if (x_index["T854T|2694T/G|T854T (2694T/G)"] == "Homozygous Mutation"):
#        value += 'D'
    
#    if ((x_index["125G/C|5UTR-8G->C"] == "No Mutation") and
#        (x_index["V470M"] == "No Mutation") and
#        (x_index["F508del|1653delCTT|[delta]F508|^F508|dF508"] == "No Mutation") and
#        (x_index["T854T|2694T/G|T854T (2694T/G)"] == "No Mutation")):
#        value += 'Z'
    
#    if ((x_index["125G/C|5UTR-8G->C"] == "No Mutation") and
#        (x_index["V470M"] == "None") and
#        (x_index["F508del|1653delCTT|[delta]F508|^F508|dF508"] == "No Mutation") and
#       (x_index["T854T|2694T/G|T854T (2694T/G)"] == "No Mutation")):
#        value += 'Z.na'
        
#    if (ancestry_no_other_df['Ancestry_Pred'][x] == 'eur'):
#        value += '_eu'

#   if (ancestry_no_other_df['Ancestry_Pred'][x] == 'amr'):
#        value += '_am'

#    if (ancestry_no_other_df['Ancestry_Pred'][x] == 'afr'):
#        value += '_af'

#    if (ancestry_no_other_df['Ancestry_Pred'][x] == 'sas'):
#        value += '_sa'
        
#    if (ancestry_no_other_df['Ancestry_Pred'][x] == 'mid'):
#        value += '_mi'
        
#    if (ancestry_no_other_df['Ancestry_Pred'][x] == 'eas'):
#        value += '_ea'
    
#    df_annotations_cropped_transposed.at[x,'category'] = value

# Check Results
#pandas.set_option('display.max_rows', None)
#pandas.set_option('display.max_columns', None)
#df_annotations_cropped_transposed


In [ ]:
## FURTHER EXTRACTING INFORMATION FOR LABELLING PLOTS DOWNSTREAM

# Obtain Pandas series for Self Reporting (Race), ancestry and ancestry (without other) categories
amount_SI = ordered_df2['race'].value_counts()
amount_Ancestry = ancestry_df['Ancestry_Pred'].value_counts()
amount_no_other_Ancestry= ancestry_no_other_df['Ancestry_Pred'].value_counts()

# Extract information for SI within Cohort
amount_White = amount_SI['White']
amount_Black = amount_SI['Black or African American']
amount_more_than_one = amount_SI['More than one population']
amount_Middle_eastern = amount_SI['Middle Eastern or North African']
amount_Asian = amount_SI['Asian']
amount_none = amount_SI['None Indicated'] + amount_SI['PMI: Skip'] + amount_SI['I prefer not to answer']+ amount_SI['None of these']

# Extract information for Ancestry within Cohort 
amount_amr = amount_Ancestry['amr']
amount_oth = amount_Ancestry['oth']
amount_afr = amount_Ancestry['afr']
amount_eur = amount_Ancestry['eur']
amount_mid = amount_Ancestry['mid']
amount_eas = amount_Ancestry['eas']
amount_sas = amount_Ancestry['sas']

# Extract information for Ancestry (no other) within Cohort: 
amount_amr_no_other = amount_no_other_Ancestry['amr']
amount_afr_no_other = amount_no_other_Ancestry['afr']
amount_eur_no_other = amount_no_other_Ancestry['eur']
amount_mid_no_other = amount_no_other_Ancestry['mid']
amount_eas_no_other = amount_no_other_Ancestry['eas']
amount_sas_no_other = amount_no_other_Ancestry['sas']

#Extract information for V470M Mutation within Cohort: 
Mutation_Value_Counts1 = df_annotations_cropped_transposed["V470M"].value_counts()
amount_No_Mutation1 = Mutation_Value_Counts1['No Mutation'] if 'No Mutation' in Mutation_Value_Counts1 else 0
amount_Heterozygous_Mutation1 = Mutation_Value_Counts1['Heterozygous Mutation'] if 'Heterozygous Mutation' in Mutation_Value_Counts1 else 0
amount_Homozygous_Mutation1 = Mutation_Value_Counts1['Homozygous Mutation'] if 'Homozygous Mutation' in Mutation_Value_Counts1 else 0
amount_NA_Mutation1 = Mutation_Value_Counts1['None'] if 'None' in Mutation_Value_Counts1 else 0

#Extract information for 125G/C|5UTR-8G->C Mutation within Cohort: 
Mutation_Value_Counts2 = df_annotations_cropped_transposed["125G/C|5UTR-8G->C"].value_counts()
amount_No_Mutation2 = Mutation_Value_Counts2['No Mutation'] if 'No Mutation' in Mutation_Value_Counts2 else 0
amount_Heterozygous_Mutation2 = Mutation_Value_Counts2['Heterozygous Mutation'] if 'Heterozygous Mutation' in Mutation_Value_Counts2 else 0
amount_Homozygous_Mutation2 = Mutation_Value_Counts2['Homozygous Mutation'] if 'Homozygous Mutation' in Mutation_Value_Counts2 else 0
amount_NA_Mutation2 = Mutation_Value_Counts2['None'] if 'None' in Mutation_Value_Counts2 else 0

#Extract information for F508del|1653delCTT|[delta]F508|^F508|dF508 Mutation within Cohort: 
Mutation_Value_Counts3 = df_annotations_cropped_transposed["F508del|1653delCTT|[delta]F508|^F508|dF508"].value_counts()
amount_No_Mutation3 = Mutation_Value_Counts3['No Mutation'] if 'No Mutation' in Mutation_Value_Counts3 else 0
amount_Heterozygous_Mutation3 = Mutation_Value_Counts3['Heterozygous Mutation'] if 'Heterozygous Mutation' in Mutation_Value_Counts3 else 0
amount_Homozygous_Mutation3 = Mutation_Value_Counts3['Homozygous Mutation'] if 'Homozygous Mutation' in Mutation_Value_Counts3 else 0
amount_NA_Mutation3 = Mutation_Value_Counts3['None'] if 'None' in Mutation_Value_Counts3 else 0

#Extract information for T854T|2694T/G|T854T (2694T/G) Mutation within Cohort: 
Mutation_Value_Counts4 = df_annotations_cropped_transposed["T854T|2694T/G|T854T (2694T/G)"].value_counts()
amount_No_Mutation4 = Mutation_Value_Counts4['No Mutation'] if 'No Mutation' in Mutation_Value_Counts4 else 0
amount_Heterozygous_Mutation4 = Mutation_Value_Counts4['Heterozygous Mutation'] if 'Heterozygous Mutation' in Mutation_Value_Counts4 else 0
amount_Homozygous_Mutation4 = Mutation_Value_Counts4['Homozygous Mutation'] if 'Homozygous Mutation' in Mutation_Value_Counts4 else 0
amount_NA_Mutation4 = Mutation_Value_Counts4['None'] if 'None' in Mutation_Value_Counts4 else 0

# Check Counts
#amount_SI
#amount_Ancestry
#amount_no_other_Ancestry
#Mutation_Value_Counts1
#Mutation_Value_Counts2
#Mutation_Value_Counts3
#Mutation_Value_Counts4

## Compare Intronic Variant prevalence between Ancestral and Racial Groups

In [ ]:
# Create a table which shows the ln variation in the prevalence of the major variant (non-mutation) 
# between self-reported White vs. self-reported Black individuals 

# Minor allele frequency is not used because if one of these two groups has a minor allele frequency of 0 it would
# the frequencies to be incomparable (undefined)

# A negative variation means that the variant is more likely to be found in Self-Defined White Individuals
# A positive variation means that the variant is more likely to be found in Self-Defined Black Individuals


black_mt = mt_filtered3.filter_cols(mt_filtered3.race.race == "Black or African American")
white_mt = mt_filtered3.filter_cols(mt_filtered3.race.race == "White")

black_mt = hl.variant_qc(black_mt)
white_mt = hl.variant_qc(white_mt)

rows_df = pandas.DataFrame(np.array(black_mt.locus.collect()))
black_df = pandas.DataFrame(np.array(black_mt.variant_qc.AF[0].collect()))
black_df = black_df.rename(columns={0: 'Black or African American'})

white_df = pandas.DataFrame(np.array(white_mt.variant_qc.AF[0].collect()))
white_df = white_df.rename(columns={0: 'White'})

comb_data1 = {
    'Locus': rows_df[0],
    'Reference Allele': pandas.DataFrame(np.array(black_mt.alleles.collect()))[0],
    'Alternate Allele': pandas.DataFrame(np.array(black_mt.alleles.collect()))[1],
    'Self-Reported White Major AF':white_df['White'],
    "Self-Reported Black or African American Major AF" : black_df['Black or African American'],
    "Variation": np.log((white_df['White'])/(black_df['Black or African American'])),
    "Variant #": pandas.Series(range(len(white_df['White'])))}

comb_df1 = pandas.DataFrame(comb_data1)
comb_df1['Locus'] = comb_df1['Locus'].astype(str)

#comb_df1 = comb_df1[(comb_df1['Variation'] <= -0.2) | (comb_df1['Variation'] >= 0.2)] - Only if you want to see
# variants with a variation above absolute(0.2)
comb_df1

In [ ]:
# Create an interactive figure which shows the above results

fig = px.bar(comb_df1, x='Variant #', y="Variation")
tickv = [-0.693, -0.6, -0.4, -0.2, 0, 0.2, 0.4, 0.6, 0.693, 0.8, 0.916, 1]
tickt = ['ln(0.5)',
         '-0.6',
         '-0.4',
         '-0.2',
         '0  ln(1)',
         '0.2',
         '0.4',
         '0.6',
        'ln(2)',
         '0.8',
        'ln(2.5)',
        '1.0']

fig.update_yaxes(tickvals = tickv, ticktext = tickt, range=[np.log(0.5),np.log(2.75)], secondary_y = False)
fig.show()

In [ ]:
# Create a table which shows the ln variation in the prevalence of the major variant (non-mutation) 
# between Genomic Defined European vs. Genomic Defined African 

# Minor allele frequency is not used because if one of these two groups has a minor allele frequency of 0 it would
# the frequencies to be incomparable (undefined)

# A negative variation means that the variant is more likely to be found in Genomic Defined European
# A positive variation means that the variant is more likely to be found in Genomic Defined African

african_mt = mt_filtered3.filter_cols(mt_filtered3.ancestry_pred.ancestry_pred == "afr")
european_mt = mt_filtered3.filter_cols(mt_filtered3.ancestry_pred.ancestry_pred == "eur")

african_mt = hl.variant_qc(african_mt)
european_mt = hl.variant_qc(european_mt)

#african_mt.variant_qc.AF.show()
#european_mt.variant_qc.AF.show()

rows_df = pandas.DataFrame(np.array(african_mt.locus.collect()))
african_df = pandas.DataFrame(np.array(african_mt.variant_qc.AF[0].collect()))
african_df = african_df.rename(columns={0: 'African'})

european_df = pandas.DataFrame(np.array(european_mt.variant_qc.AF[0].collect()))
european_df = european_df.rename(columns={0: 'European'})

comb_data2 = {
    "Locus": rows_df[0],
    'Reference Allele': pandas.DataFrame(np.array(black_mt.alleles.collect()))[0],
    'Alternate Allele': pandas.DataFrame(np.array(black_mt.alleles.collect()))[1],
    "European Major AF" : european_df['European'],
    "African Major AF" : african_df['African'],
    "Variation": np.log((european_df['European']) / (african_df['African'])),
    "Variant #": pandas.Series(range(len(african_df['African'])))}

comb_df2 = pandas.DataFrame(comb_data2)
comb_df2['Locus'] = comb_df2['Locus'].astype(str)

#comb_df1 = comb_df1[(comb_df1['Variation'] <= -0.2) | (comb_df1['Variation'] >= 0.2)] - Only if you want to see
# variants with a variation above absolute(0.2)
comb_df2

In [ ]:
# Create an interactive figure which shows the above results
fig = px.bar(comb_df2, x='Variant #', y="Variation")
tickv = [-0.693, -0.6, -0.4, -0.2, 0, 0.2, 0.4, 0.6, 0.693, 0.8, 0.916, 1]
tickt = ['ln(0.5)',
         '-0.6',
         '-0.4',
         '-0.2',
         '0  ln(1)',
         '0.2',
         '0.4',
         '0.6',
        'ln(2)',
         '0.8',
        'ln(2.5)',
        '1.0']

fig.update_yaxes(tickvals = tickv, ticktext = tickt, range=[np.log(0.5),np.log(2.75)])
fig.show()

## Self-Reported Race and Genomic Defined Ancestry Pie Charts

In [ ]:
# Creating a pie chart of the Self Reported Race of Individuals

labels = ['Black or African American', 'White', 'Middle Eastern or North African',
          'Asian', 'More than one population', 'None/Skipped']
values = [amount_Black , amount_White , amount_Middle_eastern, amount_Asian, amount_more_than_one,
         amount_none]
colors = ['black', 'white', 'green', 'orange', 'yellow', 'silver']

total = sum(values)
percentages = [f'{(value/total)*100:.1f}%' for value in values]  # Calculate percentages
legend_labels = [f'{label} (n={value}; {percentage})' for label, percentage, value in 
                 zip(labels, percentages, values)]

fig = plt.figure(facecolor='grey', figsize=(7,7))
plt.pie(values, colors = colors, startangle=90)

plt.axis('equal') 
plt.title('Self Reported Race of Individuals in the Dataset', loc='center')
plt.legend(legend_labels,loc='center', bbox_to_anchor=(0.5, -0.15))
plt.show()

In [ ]:
# Creating a pie chart of the Genomically Determined Ancestry (No "Other" Category) of Individuals

labels = ['African', 'European', 'American Admixed/Latino', 'East Asian', 'South Asian', 'Middle Eastern']
values = [amount_afr_no_other, amount_eur_no_other, amount_amr_no_other, amount_eas_no_other, amount_sas_no_other,
         amount_mid_no_other]
colors = ['black', 'white', 'violet', 'orange', 'brown', 'green']

total = sum(values)
percentages = [f'{(value/total)*100:.1f}%' for value in values]  # Calculate percentages
legend_labels = [f'{label} (n={value}; {percentage})' for label, percentage, value in 
                 zip(labels, percentages, values)]

fig = plt.figure(facecolor='silver', figsize=(7,7))
plt.pie(values, colors = colors, startangle=90)

plt.axis('equal') 
plt.title('Genomically Determined Ancestry (No "Other" Category) of Individuals in the Dataset', loc='center')
plt.legend(legend_labels,loc='center', bbox_to_anchor=(0.5, -0.15))
plt.show()

In [ ]:
# Creating a pie chart of m470v mutation in individuals 

labels = ['No Mutation', 'Heterozygous for Mutation', 'Homozygous for Mutation', 'NA']
values = [amount_No_Mutation1, amount_Heterozygous_Mutation1, amount_Homozygous_Mutation1, amount_NA_Mutation1]
colors = ['white', 'violet', 'black', 'silver']

total = sum(values)
percentages = [f'{(value/total)*100:.1f}%' for value in values]  # Calculate percentages
legend_labels = [f'{label} (n={value}; {percentage})' for label, percentage, value in 
                 zip(labels, percentages, values)]

fig = plt.figure(facecolor='grey', figsize=(7,7))
plt.pie(values, colors = colors, startangle=90)

plt.axis('equal') 
plt.title('V470M Mutation Profiling of Individuals in the Dataset', loc='center')
plt.legend(legend_labels,loc='center', bbox_to_anchor=(0.5, -0.15))
plt.show()

## Create a combined self-reported race and genomic-defined ancestry plot

In [ ]:
# Create Dataframe to sift through all combinations of self-reported race and genomic defined ancestry

comp_data2 = {
    "Self Reported Race" : ordered_df2['race'],
    "Genomic Defined Ancestry (no other)": ancestry_no_other_df['Ancestry_Pred']
}

comp_data2 = pandas.DataFrame(comp_data2)
comp_data2 = comp_data2.replace('None Indicated', 'None')
comp_data2 = comp_data2.replace('PMI: Skip', 'None')
comp_data2 = comp_data2.replace('None of these', 'None')
comp_data2 = comp_data2.replace('I prefer not to answer', 'None')
recat_counts = comp_data2.value_counts()

# Observe combinations 
recat_counts


In [ ]:
# Assign Recategorized values
white_white = recat_counts['White', 'eur']
white_admixed = recat_counts['White', 'amr']

black_black = recat_counts['Black or African American','afr']

mena_mena = recat_counts['Middle Eastern or North African', 'mid']
mena_white = recat_counts['Middle Eastern or North African', 'eur']

asian_eastasian = recat_counts['Asian','eas']
asian_southasian = recat_counts['Asian','sas']
asian_admixed = recat_counts['Asian','amr']

multi_black = recat_counts['More than one population','afr']
multi_southasian = recat_counts['More than one population','sas']

none_admixed = recat_counts['None','amr']
none_white = recat_counts['None','eur']
none_black = recat_counts['None','afr']

# check to make sure this adds up to total individual count
print(len(df_demographics) == (white_white + white_admixed + black_black + mena_mena + mena_white + 
                               asian_eastasian + asian_southasian + asian_admixed + multi_black + 
                               multi_southasian + none_admixed + none_white + none_black))

In [ ]:
# Creating a combined self-reported and genomic pie chart

fig, ax = plt.subplots()

fig.set_facecolor('white')
fig.set_size_inches(12, 18)
size = 0.3
vals1 = np.array([[white_white, white_admixed], [none_admixed,none_white,none_black], [black_black],
                  [multi_black, multi_southasian],[asian_southasian, asian_eastasian,  asian_admixed],
                  [mena_mena, mena_white]])
vals2 = np.array([white_white, white_admixed, none_admixed,none_white,none_black, black_black,
                  multi_black, multi_southasian,asian_southasian, asian_eastasian, asian_admixed,
                  mena_mena, mena_white])

outer_colors = ['red','red','silver','silver','silver',
                'black','yellow','yellow','aqua','aqua','aqua','green','green']
inner_colors = ['red','violet','violet','red','black','black','black','brown','brown','orange','violet',
               'green','red']

ax.pie(vals1.sum(axis=0), radius=1, colors=outer_colors,
       wedgeprops={'width':size, 'edgecolor':'w'})

ax.pie(vals2, radius=1-size, colors=inner_colors,
       wedgeprops={'width':size, 'edgecolor':'w'})

# Legend 1

labels1 = ['White', 'Black', 'More than one', 'Middle Eastern or North African', 'Asian', 'NA']
values1 = [amount_White, amount_Black, amount_more_than_one, amount_Middle_eastern, amount_Asian, amount_none]
colors1 = ['red', 'black', 'yellow', 'green', 'aqua', 'silver']

total1 = sum(values1)
percentages1 = [f'{(value/total1)*100:.1f}%' for value in values1]  # Calculate percentages
legend_labels1 = [f'{label} (n={value}; {percentage})' for label, percentage, value in 
                 zip(labels1, percentages1, values1)]
patches1 = [mpatches.Patch(color=color) for color in colors1]

legend1 = plt.legend(handles= patches1,labels = legend_labels1,loc='center', bbox_to_anchor=(0.275, -0.05), title =
          'Self-Reported Race (Outer Ring)')

# Legend 2
labels2 = ['European', 'African', 'Middle Eastern', 'South Asian', "East Asian", 'Admixed American']
values2 = [amount_eur_no_other, amount_afr_no_other , amount_mid_no_other, amount_sas_no_other,
           amount_eas_no_other, amount_amr_no_other]
colors2 = ['red', 'black', 'green', 'brown', 'orange', 'violet']

total2 = sum(values2)
percentages2 = [f'{(value/total2)*100:.1f}%' for value in values2]  # Calculate percentages
legend_labels2 = [f'{label} (n={value}; {percentage})' for label, percentage, value in 
                 zip(labels2, percentages2, values2)]
patches2 = [mpatches.Patch(color=color) for color in colors2]

plt.legend(handles= patches2,labels = legend_labels2,loc='center', bbox_to_anchor=(0.725, -0.05), title =
          'Genome-Defined Ancestry (Inner Ring)')

plt.gca().add_artist(legend1)

ax.set_title('Self-Reported Race vs. Genome-Defined Ancestry', y=0.95)
plt.show()



## Upload files, download and wrangle them for PCA Analysis

In [ ]:
# Upload Matrix table as TSV (and also keep one as reference)
mt_filtered3.intermediary_GT2.export(f'{workspace_bucket}/data/gt2.tsv')
mt_filtered4.ML.export(f'{workspace_bucket}/data/gt3.tsv')
mt_filtered4.alleles.export(f'{workspace_bucket}/data/alleles.tsv')
# mt_filtered3.intermediary_GT2.export(f'{workspace_bucket}/data/index_reference.tsv') - already performed, do not do it again


# Download TSV as Pandas Data Frame
df_intermediate = pandas.read_csv(f'{workspace_bucket}/data/gt2.tsv', sep= '\t')
ML_file = pandas.read_csv(f'{workspace_bucket}/data/gt3.tsv', sep= '\t')
Alleles = pandas.read_csv(f'{workspace_bucket}/data/alleles.tsv', sep= '\t')
index_reference = pandas.read_csv(f'{workspace_bucket}/data/index_reference.tsv', sep= '\t')

# Check Data Frame
#print(df.head())

# Preparing for PCA - drop the alleles column
df_drop = df_intermediate.drop('alleles',axis=1)
ML_drop = ML_file.drop('alleles',axis=1)
index_reference = index_reference.drop('alleles',axis=1)

# Set Locus column as index
df_final = df_drop.set_index('locus')
ML_final = ML_drop.set_index('locus')
index_reference = index_reference.set_index('locus')

# Turn Index_Reference into only a reference of indexes
index_reference = index_reference.transpose()
index_reference = index_reference.index

# Check Data Frame
#print(df_final.tail())

In [ ]:
df_transposed = df_final.transpose()
ML_final = ML_final.transpose().reset_index(drop=True)
ML_final['F508del'] = ''
ML_final['F508del'] = df_annotations_cropped_transposed['F508del|1653delCTT|[delta]F508|^F508|dF508']
ML_final['V470M'] = ''
ML_final['V470M'] = df_annotations_cropped_transposed['V470M']

#ML_final.to_excel(f'{workspace_bucket}/data/Genotype Files/pwCF/Intron_{intron}.xlsx', index=False)
#Alleles.to_excel(f'{workspace_bucket}/data/Genotype Files/pwCF/Intron_{intron}_Alleles.xlsx', index=False)

# Check if the index of this dataframe is equal to the index reference (Quality Control)
#if index_reference.equals(df_transposed.index):
#    print('DataFrame indices match, proceed with analysis')
#else: 
#    print('Does not match')
#print(df_transposed.head())

## PCA Analysis

### Principal Component Eigenvalues and Loading Plots 

In [ ]:
# Scikit PCA plot

# scale data
scaled_data = StandardScaler().fit_transform(df_transposed)

pca = PCA(n_components=10) if len(df_transposed.columns) >= 10 else PCA(n_components=len(df_transposed.columns))
PCA_components = pca.fit_transform(scaled_data)

# Obtain eigenvalues
eigenvalues = pca.explained_variance_

# Obtain loading values
loadings = pca.components_


In [ ]:
# Check the contribution of each principal component to variation (Scikit PCA)

x = range(1, len(eigenvalues) + 1)
plt.bar(x, eigenvalues)
plt.xlabel('Principal Component')
plt.ylabel('Eigenvalue')
plt.title('Scree Plot')
#plt.ylim(0,22.5)
plt.show()

In [ ]:
# Check the loading contribution of each variant in PC1 
loadings_pd = pandas.DataFrame(loadings)

x = list(range(1, loadings_pd.shape[1]+1))
y = abs(loadings_pd.iloc[0,:]).values.tolist()
data = [go.Bar(x=x, y=y)]

plt.figure(figsize=(10, 6))  # Adjust the figure size as per your preference
plt.bar(x, y)
plt.title('Loadings Plot for PC1')
plt.xlabel('Variant #')
plt.ylabel('Absolute Loading Value')

ax = plt.gca()
max_y_value = 0.45  
ax.set_ylim(0, max_y_value)

plt.show()


In [ ]:
# Check the loading contribution of each variant in PC2 
loadings_pd = pandas.DataFrame(loadings)

x = list(range(1, loadings_pd.shape[1]+1))
y = abs(loadings_pd.iloc[1,:]).values.tolist()
data = [go.Bar(x=x, y=y)]


plt.figure(figsize=(10, 6))  # Adjust the figure size as per your preference
plt.bar(x, y)
plt.title('Loadings Plot for PC2')
plt.xlabel('Variant #')
plt.ylabel('Absolute Loading Value')

ax = plt.gca()
max_y_value = 0.45  
ax.set_ylim(0, max_y_value)

plt.show()

In [ ]:
# Check the loading contribution of each variant in PC3
loadings_pd = pandas.DataFrame(loadings)

x = list(range(1, loadings_pd.shape[1]+1))
y = abs(loadings_pd.iloc[2,:]).values.tolist()
data = [go.Bar(x=x, y=y)]

plt.figure(figsize=(10, 6))  # Adjust the figure size as per your preference
plt.bar(x, y)
plt.title('Loadings Plot for PC3')
plt.xlabel('Variant #')
plt.ylabel('Absolute Loading Value')

ax = plt.gca()
max_y_value = 0.45  
ax.set_ylim(0, max_y_value)

plt.show()

In [ ]:
PC1 = pandas.DataFrame((loadings_pd.iloc[0,:]).values.tolist())
PC2 = pandas.DataFrame((loadings_pd.iloc[1,:]).values.tolist())
PC3 = pandas.DataFrame((loadings_pd.iloc[2,:]).values.tolist())

pcscores = pandas.concat([PC1,PC2,PC3], axis=1)

# Export if interested

### PC1-PC2

In [ ]:
# Scikit PCA plot

PC1 = PCA_components[:,0]
PC2 = PCA_components[:,1]
PC3 = PCA_components[:,2]

np.random.seed(73)

PC1_jitter = PC1 + np.random.normal(loc=0, scale=0.06*np.std(PC1), size=PC1.shape)
PC2_jitter = PC2 + np.random.normal(loc=0, scale=0.06*np.std(PC2), size=PC2.shape)
PC3_jitter = PC3 + np.random.normal(loc=0, scale=0.06*np.std(PC3), size=PC3.shape)

# Data for the first scatter plot
race_colors = {'Black or African American': 'black', 'White': 'white', 'Middle Eastern or North African': 'green',
               'Asian': 'orange', 'More than one population': 'Yellow', 'None Indicated/Skip': 'silver'}
races = ordered_df2['race']
races_labelling_colors = {'Black or African American (n={})'.format(amount_Black): 'black', 
                          'White (n={})'.format(amount_White): 'white', 
                          'Middle Eastern or North African (n={})'.format(amount_Middle_eastern): 'green',
                          'Asian (n={})'.format(amount_Asian): 'orange', 
                          'More than one population (n={})'.format(amount_more_than_one): 'Yellow',
                          'None Indicated/Skip (n={})'.format(amount_none): 'silver'}

# Data for the second scatter plot
ancestry_colors = {'afr': 'black', 'eur': 'yellow', 'amr': 'violet', 'eas': 'purple', 'sas': 'brown', 'mid': 'green', 'oth': 'silver'}
ancestries = ancestry_df['Ancestry_Pred']
ancestry_labelling_colors = {'African (n={})'.format(amount_afr): 'black', 'European (n={})'.format(amount_eur):
                             'white', 'American Admixed/Latino (n={})'.format(amount_amr): 'violet', 
                                     'East Asian (n={})'.format(amount_eas): 'orange', 'South Asian (n={})'.format(amount_sas):
                             'brown', 'Middle Eastern (n={})'.format(amount_mid): 'green',
                                      'Other (n={})'.format(amount_oth): 'silver'}

# Data for the third scatter plot
ancestry_colors = {'afr': 'black', 'eur': 'white', 'amr': 'violet', 'eas': 'orange', 'sas': 'brown', 'mid': 'green',
                   'oth': 'silver'}
ancestries_no_other = ancestry_no_other_df['Ancestry_Pred']
ancestry_no_other_labelling_colors = {'African (n={})'.format(amount_afr_no_other): 'black', 'European (n={})'.format(amount_eur_no_other):
                             'white', 'American Admixed/Latino (n={})'.format(amount_amr_no_other): 'violet', 
                                     'East Asian (n={})'.format(amount_eas_no_other): 'orange', 'South Asian (n={})'.format(amount_sas_no_other):
                             'brown', 'Middle Eastern (n={})'.format(amount_mid_no_other): 'green'}

# Create figure and axes for the subplots
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 9))

# Plot Scatter Plot 1
ax1.scatter(PC1_jitter, PC2_jitter, c=[race_colors.get(race, 'silver') for race in races], marker = 'o',
            edgecolor='black', s = 100)
ax1.set_xlabel(f"Principal Component 1 ({round((eigenvalues[0]/sum(eigenvalues))*100,1)}% of Variance)")
ax1.set_ylabel(f"Principal Component 2 ({round((eigenvalues[1]/sum(eigenvalues))*100,1)}% of Variance)")
ax1.set_title('Self Reported Race PCA Plot')
#for i, race in enumerate(races):
#    label = f'({PC1[i]:.2f}, {PC2[i]:.2f})'  # Format PC1 and PC2 values
#    ax1.text(PC1[i], PC2[i], label, color='black', fontsize='x-small')
    
# Plot Scatter Plot 2
ax2.scatter(PC1_jitter, PC2_jitter, c=[ancestry_colors.get(ancestry, 'silver') for ancestry in ancestries], marker = 'o',
            edgecolor='black', s = 100)
ax2.set_xlabel(f"Principal Component 1 ({round((eigenvalues[0]/sum(eigenvalues))*100,1)}% of Variance)")
ax2.set_ylabel(f"Principal Component 2 ({round((eigenvalues[1]/sum(eigenvalues))*100,1)}% of Variance)")
ax2.set_title('Genomically Defined Race PCA Plot')

# Plot Scatter Plot 3
ax3.scatter(PC1_jitter, PC2_jitter, c=[ancestry_colors.get(ancestry, 'silver') for ancestry in ancestries_no_other], marker = 'o',
            edgecolor='black', s = 100)
ax3.set_xlabel(f"Principal Component 1 ({round((eigenvalues[0]/sum(eigenvalues))*100,1)}% of Variance)")
ax3.set_ylabel(f"Principal Component 2 ({round((eigenvalues[1]/sum(eigenvalues))*100,1)}% of Variance)")
ax3.set_title('Genomically Defined Race PCA Plot (No "Other" Category)')

# Add the legend to the figure
race_legend_elements = [plt.Line2D([], [], linestyle='None', marker='o', markersize=8, markerfacecolor=color, markeredgecolor='black', label=label) for label, color in races_labelling_colors.items()]
ancestry_legend_elements = [plt.Line2D([], [], linestyle='None', marker='o', markersize=8, markerfacecolor=color, markeredgecolor='black', label=label) for label, color in ancestry_labelling_colors.items()]
ancestry_no_other_legend_elements = [plt.Line2D([], [], linestyle='None', marker='o', markersize=8, markerfacecolor=color, markeredgecolor='black', label=label) for label, color in ancestry_no_other_labelling_colors.items()]

fig.legend(handles=race_legend_elements, title='Self Reported Race', loc='lower center',bbox_to_anchor=
           (0.175,-0.1),ncol=3, fontsize='x-small')
fig.legend(handles=ancestry_legend_elements, title='Genomically Defined Race', loc='lower center', bbox_to_anchor=
           (0.53, -0.1), ncol=4, fontsize='x-small')
fig.legend(handles=ancestry_no_other_legend_elements, title='Genomically Defined Race (No "Other" Category)', loc='lower center', bbox_to_anchor=
           (0.85, -0.1), ncol=3, fontsize='x-small')


# Adjust spacing between subplots
plt.tight_layout()

# Save image
plt.savefig(f"Combined Race_Ancestry 2D PCA Plot Intron {intron}.pdf", format='pdf', dpi=600, bbox_inches='tight')

# Show the plot
plt.show()

### Assess differences in Within Cluster Sum of Squares (WCSS) of the Real Data with 5 artificially generated uniform dataset with the same bounds (PC1 & PC2)

In [ ]:
# Obtain Bound Box of PCs
min_pc1, max_pc1 = min(PC1), max(PC1)
min_pc2, max_pc2 = min(PC2), max(PC2)

# Set seed so results are the same everytime you run this code 
np.random.seed(0)

# Set number of points to individuals within the real dataset
points = 307

wcss_uniform = []  # store the WCSS for each number of clusters

# Create 5 uniform random reference datasets and calculate their WCSS
for point in range(5):
    x = np.random.uniform(min_pc1,max_pc1,points)
    y = np.random.uniform(min_pc2,max_pc2,points)
    dataset = np.column_stack((x,y))
    
    tempwcss = []
    max_num_clusters = 10
    
    for k in range(1, max_num_clusters+1):
        kmeans = KMeans(n_clusters=k)
        kmeans.fit(dataset)
        tempwcss.append(kmeans.inertia_)  # kmeans.inertia_ gives the WCSS value
    
    wcss_uniform.append(tempwcss)

# Calculate WCSS of Real Data
wcss = []
max_num_clusters = 10
for k in range(1, max_num_clusters+1):
    kmeans = KMeans(n_clusters=k)
    kmeans.fit(PCA_components[:,:2]) # Only take the first two PCs as uniform data only encompasses first 2 PCs
    wcss.append(kmeans.inertia_)  # kmeans.inertia_ gives the WCSS value

# Plotting the elbow curve of Uniform and Real Data
plt.plot(range(1, max_num_clusters+1), wcss_uniform[0], label = "Uniform Data 1")
plt.plot(range(1, max_num_clusters+1), wcss_uniform[1], label = "Uniform Data 2")
plt.plot(range(1, max_num_clusters+1), wcss_uniform[2], label = "Uniform Data 3")
plt.plot(range(1, max_num_clusters+1), wcss_uniform[3], label = "Uniform Data 4")
plt.plot(range(1, max_num_clusters+1), wcss_uniform[4], label = "Uniform Data 5")
plt.plot(range(1, max_num_clusters+1), wcss, label = "Real Data")
plt.xlabel('Number of Clusters')
plt.ylabel('WCSS')
plt.title('Elbow Method (PCA)')
plt.legend()
plt.show()


In [ ]:
## Compare the difference in the results of the previous plot to assess at what cluster the difference in WCSS is
## the most

# Calculating the difference between the Uniform and Real Data
Wcss_df = pandas.DataFrame(wcss_uniform)
Wcss_df = Wcss_df.transpose()
Wcss_df["Average_Uniform"]= Wcss_df.mean(axis=1)
Wcss_df["RealData"]= wcss
Wcss_df["Difference"] = Wcss_df["RealData"] - Wcss_df["Average_Uniform"]
Diff_list = Wcss_df["Difference"].tolist()

# Plotting the difference
plt.plot(range(1, max_num_clusters+1), Diff_list)
plt.xlabel('Number of Clusters')
plt.ylabel('WCSS difference between Real and Uniform Data')
plt.title('Gap Statistics WCSS PCA')
plt.show()



### Assess differences in Silhouette Scores of the Real Data with 5 artificially generated uniform dataset with the same bounds (PC1 & PC2)

In [ ]:
# Set bounds for the uniform data
min_pc1, max_pc1 = min(PC1), max(PC1)
min_pc2, max_pc2 = min(PC2), max(PC2)

# Set seed so results are the same everytime you run this code 
np.random.seed(0)

# Set number of points to individuals within the real dataset
points = 307

# store the silhouette score for each number of clusters in the uniform data
silhouette_uniform = []  

# Create 5 uniform random reference datasets
for point in range(5):
    x = np.random.uniform(min_pc1,max_pc1,points)
    y = np.random.uniform(min_pc2,max_pc2,points)
    dataset = np.column_stack((x,y))
    
    tempsilhouette = []
    max_num_clusters = 10
    
    for k in range(2, max_num_clusters+1):
        kmeans = KMeans(n_clusters=k, random_state = 0)
        labels = kmeans.fit_predict(dataset)
        silhouette_avg = silhouette_score(dataset, labels)
        tempsilhouette.append(silhouette_avg)
    
    silhouette_uniform.append(tempsilhouette)

# store the Silhouette Score each number of clusters in the real data
silhouette_scores = [] 
max_num_clusters = 10  

for k in range(2, max_num_clusters+1):
    kmeans = KMeans(n_clusters=k, random_state = 0)
    labels = kmeans.fit_predict(PCA_components[:,:2]) # Only take the first two PCs as uniform data only encompasses first 2 PCs
    silhouette_avg = silhouette_score(PCA_components[:,:2], labels)
    silhouette_scores.append(silhouette_avg) 

# Plotting the Silhouette Score curve
plt.plot(range(2, max_num_clusters+1), silhouette_scores, label = "Real Data")
plt.plot(range(2, max_num_clusters+1), silhouette_uniform[0], label = "Uniform Data 1")
plt.plot(range(2, max_num_clusters+1), silhouette_uniform[1], label = "Uniform Data 2")
plt.plot(range(2, max_num_clusters+1), silhouette_uniform[2], label = "Uniform Data 3")
plt.plot(range(2, max_num_clusters+1), silhouette_uniform[3], label = "Uniform Data 4")
plt.plot(range(2, max_num_clusters+1), silhouette_uniform[4], label = "Uniform Data 5")
plt.xlabel('Number of Clusters')
plt.ylabel('silhouette_score')
plt.title('Silhouette Score for PCA')
plt.legend()
plt.show()

In [ ]:
# Calculating the difference between the uniform and the real data
silhouettePCA_df = pandas.DataFrame(silhouette_uniform)
silhouettePCA_df = silhouettePCA_df.transpose()
silhouettePCA_df["Average_Uniform"]= silhouettePCA_df.mean(axis=1)
silhouettePCA_df["RealData"]= silhouette_scores
silhouettePCA_df["Difference"] = silhouettePCA_df["RealData"] - silhouettePCA_df["Average_Uniform"]
Diff_listPCA = silhouettePCA_df["Difference"].tolist()

# Calculating the plot 
plt.plot(range(2, max_num_clusters+1), Diff_listPCA)
plt.xlabel('Number of Clusters')
plt.ylabel('Silhouette Score Difference between Real and Uniform Data')
plt.title('Gap Statistics Silhoutte Score PCA')
plt.show()



### Visualization of Clustering based on KMeans (Optional)

In [ ]:
kmeans = KMeans(n_clusters=6, random_state = 0)
labels = kmeans.fit_predict(PCA_components[:,:2])

# Create a scatter plot of the data points
plt.figure(figsize=(9,9))
plt.scatter(PCA_components[:, 0], PCA_components[:, 1], c=labels, cmap='viridis')

# Labelling
plt.xlabel('Dimension 1')
plt.ylabel('Dimension 2')
plt.title('Kmeans Clustering Visualization for PCA')

# Plot
plt.show()

# Calculate Silhouette Score
silhouette_avg = silhouette_score(PCA_components[:,:2], labels)
print("The silhouette score is:", silhouette_avg)


### PCA with Sex Labelling

In [ ]:
# 2D PCA based on Sex

# Data for scatter plot
mutation_colors = {'Male': 'blue', 'Female': 'orange'}

# Create figure and axes for the subplots
fig,ax = plt.subplots(figsize=(9,9))

# Plot Scatter Plot
ax.scatter(PC1_jitter, PC2_jitter, c=[mutation_colors.get(sex, 'red') for sex in df_demographics['sex_at_birth']], marker = 'o',
            edgecolor='black')
ax.set_xlabel(f"Principal Component 1 ({round((eigenvalues[0]/sum(eigenvalues))*100,1)}% of Variance)")
ax.set_ylabel(f"Principal Component 2 ({round((eigenvalues[1]/sum(eigenvalues))*100,1)}% of Variance)")
ax.set_title(f"2D PCA for Intron {intron} Variation in pwCF profiled based on their sex at birth")


# Add the legend to the figure
mutation_legend_elements = [plt.Line2D([], [], linestyle='None', marker='o', markersize=8, markerfacecolor=color, markeredgecolor='black', label=label) for label, color in mutation_colors.items()]

# Adjust spacing between subplots
plt.tight_layout()

# Show the plot
plt.show()

In [ ]:
# 3D PCA based on sex
fig = plt.figure(figsize=(11.5,12))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(PC1_jitter, PC2_jitter, PC3_jitter, c=[mutation_colors.get(sex, 'red') for sex in df_demographics['sex_at_birth']], marker = 'o',
            edgecolor='black')

mutation_legend_elements = [plt.Line2D([], [], linestyle='None', marker='o', markersize=8, markerfacecolor=color, markeredgecolor='black', label=label) for label, color in mutation_colors.items()]

ax.set_xlabel(f"Principal Component 1 ({round((eigenvalues[0]/sum(eigenvalues))*100,1)}% of Variance)", fontsize = 11)
ax.set_ylabel(f"Principal Component 2 ({round((eigenvalues[1]/sum(eigenvalues))*100,1)}% of Variance)", fontsize = 11)
ax.set_zlabel(f"Principal Component 3 ({round((eigenvalues[2]/sum(eigenvalues))*100,1)}% of Variance)", fontsize = 11)
ax.set_title(f"3D PCA for Intron {intron} Variation in pwCF profiled based on their sex at birth", y=1, fontsize=15)
ax.dist = 10.175

ax.set_xlim(left=min(PC1_jitter), right=max(PC1_jitter))
ax.set_ylim(bottom=min(PC2_jitter), top=max(PC2_jitter))
ax.set_zlim(bottom=min(PC3_jitter), top=max(PC3_jitter))

# Save Image
plt.savefig(f"Sex at Birth 3D PCA Plot Intron {intron}.pdf", format='pdf', dpi=600, bbox_inches='tight')

### PCA with V470M Mutation Labelling

In [ ]:
# 2D PCA based on V470M mutation

# Data for scatter plot
mutation_colors = {'No Mutation': 'white', 'Heterozygous Mutation': 'violet', 'Homozygous Mutation': 'black', 'None': 'red'}
mutation_labelling_colors = {'No Mutation (n={})'.format(amount_No_Mutation1): 'white', 'Heterozygous for Mutation (n={})'.format(amount_Heterozygous_Mutation1):
                             'violet', 'Homozygous for Mutation (n={})'.format(amount_Homozygous_Mutation1): 'black', 
                                     'NA (n={})'.format(amount_NA_Mutation1): 'red'}

# Create figure and axes for the subplots
fig,ax = plt.subplots(figsize=(9, 9))

# Plot Scatter Plot
ax.scatter(PC1_jitter, PC2_jitter, c=[mutation_colors.get(mutation, 'red') for mutation in df_annotations_cropped_transposed["V470M"]], marker = 'o',
            edgecolor='black')
ax.set_xlabel(f"Principal Component 1 ({round((eigenvalues[0]/sum(eigenvalues))*100,1)}% of Variance)")
ax.set_ylabel(f"Principal Component 2 ({round((eigenvalues[1]/sum(eigenvalues))*100,1)}% of Variance)")
ax.set_title(f"2D PCA for Intron {intron} Variation in pwCF profiled based on V470M")


# Add the legend to the figure
mutation_legend_elements = [plt.Line2D([], [], linestyle='None', marker='o', markersize=8, markerfacecolor=color, markeredgecolor='black', label=label) for label, color in mutation_labelling_colors.items()]


# Adjust spacing between subplots
plt.tight_layout()

# Show the plot
plt.show()

In [ ]:
# 3D PCA for V470M
fig = plt.figure(figsize=(11.5, 12))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(PC1_jitter, PC2_jitter, PC3_jitter, c=[mutation_colors.get(mutation, 'red') for mutation in df_annotations_cropped_transposed["V470M"]], marker = 'o',
            edgecolor='black', s = 60)

mutation_legend_elements = [plt.Line2D([], [], linestyle='None', marker='o', markersize=8, markerfacecolor=color, markeredgecolor='black', label=label) for label, color in mutation_labelling_colors.items()]

ax.set_xlabel(f"Principal Component 1 ({round((eigenvalues[0]/sum(eigenvalues))*100,1)}% of Variance)", fontsize = 11)
ax.set_ylabel(f"Principal Component 2 ({round((eigenvalues[1]/sum(eigenvalues))*100,1)}% of Variance)", fontsize = 11)
ax.set_zlabel(f"Principal Component 3 ({round((eigenvalues[2]/sum(eigenvalues))*100,1)}% of Variance)", fontsize = 11)
ax.set_title(f"3D PCA for Intron {intron} Variation in pwCF profiled based on the presence of V470M mutation", y=1, fontsize=15)

ax.set_xlim(left=min(PC1_jitter), right=max(PC1_jitter))
ax.set_ylim(bottom=min(PC2_jitter), top=max(PC2_jitter))
ax.set_zlim(bottom=min(PC3_jitter), top=max(PC3_jitter))

# save image
plt.savefig(f"V470M 3D PCA Plot Intron {intron}.pdf", format='pdf', dpi=600, bbox_inches='tight')



### PCA with F508del Mutation Labelling

In [ ]:
# 2D PCA based on F508del mutation

# Data for scatter plot
mutation_colors = {'No Mutation': 'white', 'Heterozygous Mutation': 'violet', 'Homozygous Mutation': 'black', 'None': 'red'}
mutation_labelling_colors = {'No Mutation (n={})'.format(amount_No_Mutation1): 'white', 'Heterozygous for Mutation (n={})'.format(amount_Heterozygous_Mutation1):
                             'violet', 'Homozygous for Mutation (n={})'.format(amount_Homozygous_Mutation1): 'black', 
                                     'NA (n={})'.format(amount_NA_Mutation1): 'red'}

# Create figure and axes for the subplots
fig,ax = plt.subplots(figsize=(9, 9))

# Plot Scatter Plot 3
ax.scatter(PC1_jitter, PC2_jitter, c=[mutation_colors.get(mutation, 'red') for mutation in df_annotations_cropped_transposed["F508del|1653delCTT|[delta]F508|^F508|dF508"]], marker = 'o',
            edgecolor='black')
ax.set_xlabel(f"Principal Component 1 ({round((eigenvalues[0]/sum(eigenvalues))*100,1)}% of Variance)")
ax.set_ylabel(f"Principal Component 2 ({round((eigenvalues[1]/sum(eigenvalues))*100,1)}% of Variance)")
ax.set_title(f"2D PCA for Intron {intron} Variation in pwCF profiled based on the presence of F508del mutation")


# Add the legend to the figure

mutation_legend_elements = [plt.Line2D([], [], linestyle='None', marker='o', markersize=8, markerfacecolor=color, markeredgecolor='black', label=label) for label, color in mutation_labelling_colors.items()]


# Adjust spacing between subplots
plt.tight_layout()

# Show the plot
plt.show()

In [ ]:
# 3D PCA for F508del
fig = plt.figure(figsize=(11.5, 12))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(PC1_jitter, PC2_jitter, PC3_jitter, c=[mutation_colors.get(mutation, 'red') for mutation in df_annotations_cropped_transposed["F508del|1653delCTT|[delta]F508|^F508|dF508"]], marker = 'o',
            edgecolor='black', s = 60)

mutation_legend_elements = [plt.Line2D([], [], linestyle='None', marker='o', markersize=8, markerfacecolor=color, markeredgecolor='black', label=label) for label, color in mutation_labelling_colors.items()]

ax.set_xlabel(f"Principal Component 1 ({round((eigenvalues[0]/sum(eigenvalues))*100,1)}% of Variance)", fontsize = 11)
ax.set_ylabel(f"Principal Component 2 ({round((eigenvalues[1]/sum(eigenvalues))*100,1)}% of Variance)", fontsize = 11)
ax.set_zlabel(f"Principal Component 3 ({round((eigenvalues[2]/sum(eigenvalues))*100,1)}% of Variance)", fontsize = 11)
ax.set_title(f"3D PCA for Intron {intron} Variation in pwCF profiled based on the presence of F508del mutation", y=1, fontsize=15)

ax.set_xlim(left=min(PC1_jitter), right=max(PC1_jitter))
ax.set_ylim(bottom=min(PC2_jitter), top=max(PC2_jitter))
ax.set_zlim(bottom=min(PC3_jitter), top=max(PC3_jitter))

# save image
plt.savefig(f"F508del 3D PCA Plot Intron {intron}.pdf", format='pdf', dpi=600, bbox_inches='tight')


### PCA with CFTR Variant Labelling

In [ ]:
# PCA based on CF Status

# Data for scatter plot
mutation_colors = {'Non-CF': 'white', 'CF': 'black', 'None': 'silver'}
mutation_labelling_colors2 = {'No CFTR Variants (n={})'.format(len(List_E)):
                              {'color':'white', 'edgecolor':'green'},
                              'Non-Pathogenic CFTR Variants or VUS (n={})'.format(len(List_D)):
                              {'color':'white', 'edgecolor':'purple'}, 
                              'F508del Homozygous (n={})'.format(len(List_C)):
                              {'color':'black', 'edgecolor':'orange'},
                              'F508del/CF-Causing | F508del/Varying Clinical Significance (n={})'.format(len(List_A)):
                              {'color':'black', 'edgecolor':'red'},
                              'CF-Causing Homozygous | CF-Causing/Varying Clinical Significance (n={})'.format(len(List_B)):
                              {'color':'black', 'edgecolor':'lightblue'}}


# Create figure and axes for the subplots
plt.figure(figsize=(9, 9))

# Plot Scatter Plot 3
plt.scatter(PC1_jitter[List_D], PC2_jitter[List_D],
            c=[mutation_colors.get(mutation, 'red') for mutation in individual_mutations['CF Designation'][List_D]], marker = 'o',
            edgecolor='purple')
plt.scatter(PC1_jitter[List_A], PC2_jitter[List_A],
            c=[mutation_colors.get(mutation, 'red') for mutation in individual_mutations['CF Designation'][List_A]], marker = 'o',
            edgecolor='red')
plt.scatter(PC1_jitter[List_B], PC2_jitter[List_B],
            c=[mutation_colors.get(mutation, 'red') for mutation in individual_mutations['CF Designation'][List_B]], marker = 'o',
            edgecolor='lightblue')
plt.scatter(PC1_jitter[List_C], PC2_jitter[List_C],
            c=[mutation_colors.get(mutation, 'red') for mutation in individual_mutations['CF Designation'][List_C]], marker = 'o',
            edgecolor='gold')
plt.scatter(PC1_jitter[List_E], PC2_jitter[List_E],
            c=[mutation_colors.get(mutation, 'red') for mutation in individual_mutations['CF Designation'][List_E]], marker = 'o',
            edgecolor='green')

plt.xlabel(f"Principal Component 1 ({round((eigenvalues[0]/sum(eigenvalues))*100,1)}% of Variance)")
plt.ylabel(f"Principal Component 2 ({round((eigenvalues[1]/sum(eigenvalues))*100,1)}% of Variance)")
plt.title(f"2D PCA for Intron {intron} Variation in pwCF profiled based on CFTR Variants")

# Add the legend to the figure


mutation_legend_elements2 = [plt.Line2D([], [], linestyle='None', marker='o', markersize=8, markerfacecolor=color,
                                       markeredgecolor=edgecolor, label=label) for label,
                            info in mutation_labelling_colors2.items() for color,
                            edgecolor in [(info['color'], info['edgecolor'])]]


#plt.legend(handles=mutation_legend_elements2, loc='lower center', title='CFTR Variants', fontsize='9',bbox_to_anchor=(0.5, -0.3), ncol=1)

# Adjust spacing between subplots
plt.tight_layout()

# Show the plot
plt.show()

In [ ]:
# 3D PCA based on CF Status

# Data for scatter plot
mutation_colors = {'Non-CF': 'white', 'CF': 'black', 'None': 'silver'}
mutation_labelling_colors2 = {'No CFTR Variants (n={})'.format(len(List_E)):
                              {'color':'white', 'edgecolor':'green'},
                              'Non-Pathogenic CFTR Variants or VUS (n={})'.format(len(List_D)):
                              {'color':'white', 'edgecolor':'purple'}, 
                              'F508del Homozygous (n={})'.format(len(List_C)):
                              {'color':'black', 'edgecolor':'orange'},
                              'F508del/CF-Causing | F508del/Varying Clinical Significance (n={})'.format(len(List_A)):
                              {'color':'black', 'edgecolor':'red'},
                              'CF-Causing Homozygous | CF-Causing/Varying Clinical Significance (n={})'.format(len(List_B)):
                              {'color':'black', 'edgecolor':'lightblue'}}


# Create figure and axes for the subplots
fig = plt.figure(figsize=(11.5, 12))
ax = fig.add_subplot(111, projection='3d')

# Plot Scatter Plot 3
ax.scatter(PC1_jitter[List_D], PC2_jitter[List_D], PC3_jitter[List_D],
           c=[mutation_colors.get(mutation, 'red') for mutation in individual_mutations['CF Designation'][List_D]], marker = 'o',
            edgecolor='purple', s = 90)
ax.scatter(PC1_jitter[List_A], PC2_jitter[List_A], PC3_jitter[List_A],
            c=[mutation_colors.get(mutation, 'red') for mutation in individual_mutations['CF Designation'][List_A]], marker = 'o',
            edgecolor='red', s = 90)
ax.scatter(PC1_jitter[List_B], PC2_jitter[List_B], PC3_jitter[List_B],
            c=[mutation_colors.get(mutation, 'red') for mutation in individual_mutations['CF Designation'][List_B]], marker = 'o',
            edgecolor='lightblue', s = 90)
ax.scatter(PC1_jitter[List_C], PC2_jitter[List_C], PC3_jitter[List_C],
            c=[mutation_colors.get(mutation, 'red') for mutation in individual_mutations['CF Designation'][List_C]], marker = 'o',
            edgecolor='gold', s = 90)
ax.scatter(PC1_jitter[List_E], PC2_jitter[List_E], PC3_jitter[List_E],
            c=[mutation_colors.get(mutation, 'red') for mutation in individual_mutations['CF Designation'][List_E]], marker = 'o',
            edgecolor='green', s = 90)

ax.set_xlabel(f"Principal Component 1 ({round((eigenvalues[0]/sum(eigenvalues))*100,1)}% of Variance)",fontsize = 13)
ax.set_ylabel(f"Principal Component 2 ({round((eigenvalues[1]/sum(eigenvalues))*100,1)}% of Variance)", fontsize = 13)
ax.set_zlabel(f"Principal Component 3 ({round((eigenvalues[2]/sum(eigenvalues))*100,1)}% of Variance)", fontsize = 13)
ax.set_title(f"3D PCA for Intron {intron} Variation in pwCF profiled based on CFTR Variants", y=1, fontsize=17.5)

ax.set_xlim(left=min(PC1_jitter), right=max(PC1_jitter))
ax.set_ylim(bottom=min(PC2_jitter), top=max(PC2_jitter))
ax.set_zlim(bottom=min(PC3_jitter), top=max(PC3_jitter))
# Add the legend to the figure


mutation_legend_elements2 = [plt.Line2D([], [], linestyle='None', marker='o', markersize=8, markerfacecolor=color,
                                       markeredgecolor=edgecolor, label=label) for label,
                            info in mutation_labelling_colors2.items() for color,
                            edgecolor in [(info['color'], info['edgecolor'])]]


#plt.legend(handles=mutation_legend_elements2, loc='lower center', title='CFTR Variants', fontsize='9',bbox_to_anchor=(0.5, -0.3), ncol=1)

# Adjust spacing between subplots
plt.tight_layout()

# save image
plt.savefig(f"CFTR Variants 3D PCA Plot Intron {intron}.pdf", format='pdf', dpi=600, bbox_inches='tight')

# Show the plot
plt.show()

### Loadings Plot

In [ ]:
# Loadings Plot 

labels = df_transposed.columns
plt.figure(figsize=(9,9))
plt.scatter(loadings[0],loadings[1], label = labels)
plt.title("Loadings Plot")
plt.xlabel(f"Principal Component 1 ({round((eigenvalues[0]/sum(eigenvalues))*100,1)}% of Variance)")
plt.ylabel(f"Principal Component 2 ({round((eigenvalues[1]/sum(eigenvalues))*100,1)}% of Variance)")

for i, label in enumerate(labels):
    plt.annotate(label, (loadings[0][i], loadings[1][i]), textcoords="offset points", xytext=(0, 0), ha="center",
                fontsize = 6)

plt.axhline(0, color='gray', dashes=(2, 2))
plt.axvline(0, color='gray', dashes=(2, 2))

plt.tight_layout()
plt.show()

### Bi-Plot

In [ ]:
# Bi-Plot 

#labels = df_transposed.columns
labels = list(range(1,33))

# Data for scatter plot
mutation_colors = {'Non-CF': 'white', 'CF': 'black', 'None': 'silver'}
mutation_labelling_colors = {'No Mutation (n={})'.format(individual_mutations['CF Designation'].value_counts()['Non-CF']): 'white',
                             'CF (n={})'.format(individual_mutations['CF Designation'].value_counts()['CF']): 'black'}

# Plot Scatter Plots 
fig, ax1 =  plt.subplots(figsize=(9,9))
tempax2 = ax1.twinx()
ax2 = tempax2.twiny()

ax1.scatter(PC1_jitter[List_D], PC2_jitter[List_D],
            c=[mutation_colors.get(mutation, 'silver') for mutation in individual_mutations['CF Designation'][List_D]], marker = 'o',
            edgecolor='purple')
ax1.scatter(PC1_jitter[List_A], PC2_jitter[List_A],
            c=[mutation_colors.get(mutation, 'silver') for mutation in individual_mutations['CF Designation'][List_A]], marker = 'o',
            edgecolor='red')
ax1.scatter(PC1_jitter[List_B], PC2_jitter[List_B],
            c=[mutation_colors.get(mutation, 'silver') for mutation in individual_mutations['CF Designation'][List_B]], marker = 'o',
            edgecolor='lightblue')
ax1.scatter(PC1_jitter[List_C], PC2_jitter[List_C],
            c=[mutation_colors.get(mutation, 'silver') for mutation in individual_mutations['CF Designation'][List_C]], marker = 'o',
            edgecolor='gold')
ax1.scatter(PC1_jitter[List_E], PC2_jitter[List_E],
            c=[mutation_colors.get(mutation, 'silver') for mutation in individual_mutations['CF Designation'][List_E]], marker = 'o',
            edgecolor='green')

ax1.set_xlabel(f"Principal Component 1 Scores ({round((eigenvalues[0]/sum(eigenvalues))*100,1)}% of Variance)")
ax1.set_ylabel(f"Principal Component 2 Scores ({round((eigenvalues[1]/sum(eigenvalues))*100,1)}% of Variance)")

np.random.seed(73)
loading_0_jitter = loadings[0] * np.random.uniform(0.97,1.03, size = loadings[1].shape[0])
loading_1_jitter = loadings[1] * np.random.uniform(0.97,1.03, size = loadings[1].shape[0])
ax2.scatter(loading_0_jitter,loading_1_jitter, label = labels, color='blue')
ax2.set_xlabel(f"Principal Component 1 Variable Loadings", color = 'blue', labelpad = 11)
tempax2.set_ylabel(f"Principal Component 2 Variable Loadings", color = 'blue', labelpad = 11)
ax2.tick_params(axis = 'x', colors='blue')
tempax2.tick_params(axis = 'y', colors='blue')

# Data for scatter plot
mutation_colors = {'Non-CF': 'white', 'CF': 'black', 'None': 'silver'}
mutation_labelling_colors = {'Non-CF (n={})'.format(individual_mutations['CF Designation'].value_counts()['Non-CF']): 'white',
                             'CF (n={})'.format(individual_mutations['CF Designation'].value_counts()['CF']): 'black'}

ax1.axhline(0, color='gray', dashes=(2, 2))
ax1.axvline(0, color='gray', dashes=(2, 2))

# Set up font
font = {'color': 'g',
       'weight': 'bold',
       'size': '5'}

for i in range(len(loadings[0])):
    ax2.arrow(0,0, loading_0_jitter[i], loading_1_jitter[i], alpha = 0.15, color = 'grey')
    #ax2.text(loading_0_jitter[i],loading_1_jitter[i], labels[i], ha = 'center', va = 'center', fontdict = font)
# Adjust spacing between subplots
plt.tight_layout()
plt.title('Bi-Plot', weight='bold')

# Align axes 
mpl_axes_aligner.align.xaxes(ax1, 0, ax2, 0, 0.5)
mpl_axes_aligner.align.yaxes(ax1, 0, ax2, 0, 0.5)

# save image
plt.savefig(f"Biplot Intron {intron}.pdf", format='pdf', dpi=600, bbox_inches='tight')

# Show the plot
plt.show()



### PCA labelled by ancestry and the 4 top CFTR variants in conjunction

In [ ]:
# Create figure and axes for the subplots
fig, ax = plt.subplots(figsize=(18,18))

# Plot Scatter Plot 3
for i, category in enumerate(df_annotations_cropped_transposed['category']):
    ax.text(PC1[i], PC2[i], category, fontsize=12, color='black', ha='center', va='center')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.scatter(PCA_components[:, 0], PCA_components[:, 1], cmap='rainbow')
plt.title(f"PCA for Intron {intron} Variation in pwCF profiled based on the presence of mutation")

ax.set_xlim(left=min(PC1-0.2), right=max(PC1+0.2))
ax.set_ylim(bottom=min(PC2-0.2), top=max(PC2+0.2))
# Adjust s

# Show the plot
fig.show()

### PC2-PC3

In [ ]:
# PC2 and PC3
PC3 = PCA_components[:,2]

# Data for the third scatter plot
ancestry_colors = {'afr': 'black', 'eur': 'white', 'amr': 'violet', 'eas': 'orange', 'sas': 'brown', 'mid': 'green',
                   'oth': 'silver'}
ancestries_no_other = ancestry_no_other_df['Ancestry_Pred']
ancestry_no_other_labelling_colors = {'African (n={})'.format(amount_afr_no_other): 'black', 'European (n={})'.format(amount_eur_no_other):
                             'white', 'American Admixed/Latino (n={})'.format(amount_amr_no_other): 'violet', 
                                     'East Asian (n={})'.format(amount_eas_no_other): 'orange', 'South Asian (n={})'.format(amount_sas_no_other):
                             'brown', 'Middle Eastern (n={})'.format(amount_mid_no_other): 'green'}

# Create figure and axes for the subplots
plt.figure(figsize=(9, 9))

# Plot Scatter Plot 3
plt.scatter(PC2, PC3, c=[ancestry_colors.get(ancestry, 'silver') for ancestry in ancestries_no_other], marker = 'o',
            edgecolor='black')
plt.xlabel('Principal Component 2')
plt.ylabel('Principal Component 3')
plt.title('Genomically Defined Race PCA Plot (No "Other" Category)')

# Add the legend to the figure
race_legend_elements = [plt.Line2D([], [], linestyle='None', marker='o', markersize=8, markerfacecolor=color, markeredgecolor='black', label=label) for label, color in races_labelling_colors.items()]
ancestry_legend_elements = [plt.Line2D([], [], linestyle='None', marker='o', markersize=8, markerfacecolor=color, markeredgecolor='black', label=label) for label, color in ancestry_labelling_colors.items()]
ancestry_no_other_legend_elements = [plt.Line2D([], [], linestyle='None', marker='o', markersize=8, markerfacecolor=color, markeredgecolor='black', label=label) for label, color in ancestry_no_other_labelling_colors.items()]

plt.legend(handles=ancestry_no_other_legend_elements, title='Genomically Defined Race (No "Other" Category)', loc='lower center', bbox_to_anchor=
           (0.5, -0.17), ncol=3, fontsize='x-small')

# Adjust spacing between subplots
plt.tight_layout()

# Show the plot
plt.show()